<a href="https://colab.research.google.com/github/minyi-k03/LargeLanguageModel/blob/Project-Based-Learning(PBL)/Langchain_Quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangChain Quickstart 기초예제
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## Reference : https://python.langchain.com/docs/get_started/quickstart

# LangChain 라이브러리 설치

In [ ]:
!pip install -q -U langchain langchain-openai langchain-community openai

print("설치가 완료되었습니다!")

# OpenAI 라이브러리 설정

In [ ]:
!pip install openai

# OpenAI API를 이용한 LLM 설정



*   **LLMs** : string을 input으로 받아서 string을 return하는 모듈
*   **ChatModels** : message 리스트를 받아서 message 리스트를 return하는 모듈



In [ ]:
OPENAI_KEY = "Input Your Key"

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(openai_api_key=OPENAI_KEY) # 기본모델 : text-davinci-003

In [ ]:
response = llm.invoke("안녕!")

print(response.content)

In [ ]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(
    api_key=OPENAI_KEY,
    model="gpt-4o-mini",
)


In [ ]:
response = chat_model.invoke("안녕!")
print(response.content)

*   **predict** : string을 input으로 받아서 string을 return
*   **predict_messages** : message 리스트를 받아서 message 리스트를 return

In [ ]:
text = "컬러풀 양말을 만드는 회사의 좋은 이름은 무엇일까요?"

llm.invoke(text)

In [ ]:
chat_model.invoke(text)

*   **HumanMessage** : 사람으로부터 주어진 message
*   **AIMessage** : AI/assistant로부터 주어진 message
*   **SystemMessage** : system으로부터 주어진 message
*   **FunctionMessage** : function call로부터 주어진 message

In [ ]:
from langchain_core.messages import HumanMessage

text = "컬러풀 양말을 만드는 회사의 좋은 이름은 무엇일까요?"
messages = [HumanMessage(content=text)]

response = chat_model.invoke(messages)

print(response.content)

In [ ]:
chat_model.invoke(messages)

# 프롬프트 템플릿(Prompt templates)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("{product}을 만드는 회사의 좋은 이름은 무엇일까요?")

formatted_text = prompt.format(product="컬러풀 양말")

print(formatted_text)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = "당신은 도움이 되는 조수입니다. {input_language}을 {output_language}로 번역하세요."
human_template = "{text}"

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("user", human_template), # 'human' 대신 'user'라고 써도 됩니다.
])

messages = chat_prompt.format_messages(
    input_language="English",
    output_language="Korean",
    text="I love programming."
)

print("[생성된 메시지 확인]:")
print(messages)

# Output parsers

In [ ]:
from langchain_core.output_parsers import BaseOutputParser
from typing import List

class CommaSeparatedListOutputParser(BaseOutputParser[List[str]]):
    """LLM의 출력을 콤마로 구분된 리스트로 변환하는 파서"""

    def parse(self, text: str) -> List[str]:
        """텍스트를 받아서 리스트로 반환"""
        return text.strip().split(", ")

parser = CommaSeparatedListOutputParser()
result = parser.parse("hi, bye")

print("[결과 타입]:", type(result))
print("[결과 값]:", result)

# PromptTemplate + LLM + OutputParser

In [ ]:
template = """당신은 쉼표로 구분된 목록을 생성하는 유용한 조수입니다. \
사용자가 카테고리를 전달하면 해당 카테고리에 속하는 5개의 객체를 쉼표로 구분된 목록으로 생성합니다. \
오직 쉼표로 구분된 목록만 반환하고 그 이상은 반환하지 마세요."""
human_template = "{text}"

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", human_template),
])
chain = chat_prompt | chat_model | CommaSeparatedListOutputParser()
chain.invoke({"text": "색깔"})